# PS-003 Workforce Growth Analysis

This notebook reviews the exported growth-rate artefact, surfaces the top profession-level CAGR values, inspects nurse year-on-year growth, and saves the static figures required for the dashboard handoff.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import polars as pl

workspace_root = next(
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / 'shared').exists() and (path / 'docs').exists()
)
src_dir = workspace_root / 'artifacts' / 'ps-003-workforce-growth-analysis' / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

growth_rates_path = workspace_root / 'artifacts' / 'ps-003-workforce-growth-analysis' / 'results' / 'tables' / 'growth_rates.parquet'
figures_dir = workspace_root / 'artifacts' / 'ps-003-workforce-growth-analysis' / 'reports' / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)
cagr_figure_path = figures_dir / 'cagr_by_profession.png'
yoy_figure_path = figures_dir / 'yoy_growth_by_profession.png'

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
growth_rates_path

PosixPath('/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/artifacts/ps-003-workforce-growth-analysis/results/tables/growth_rates.parquet')

## Load Growth Rates

Load the exported parquet and confirm the schema, row count, and a sample of records before deriving any visual summaries.

In [2]:
growth_df = pl.read_parquet(growth_rates_path)
print('Shape:', growth_df.shape)
print('Schema:', growth_df.schema)
growth_df.head(10)

Shape: (192, 5)
Schema: Schema([('profession', Categorical), ('sector', Categorical), ('year', Int32), ('yoy_pct', Float64), ('cagr', Float64)])


profession,sector,year,yoy_pct,cagr
cat,cat,i32,f64,f64
"""doctors""","""All""",2006,null,0.057174
"""doctors""","""All""",2007,7.690088,0.057174
"""doctors""","""All""",2008,5.050911,0.057174
"""doctors""","""All""",2009,6.147175,0.057174
"""doctors""","""All""",2010,8.494533,0.057174
"""doctors""","""All""",2011,6.821705,0.057174
"""doctors""","""All""",2012,6.002488,0.057174
"""doctors""","""All""",2013,7.119804,0.057174
"""doctors""","""All""",2014,7.121337,0.057174


## Top CAGR Values By Profession

Profession-level CAGR is stored on the sector = 'All' aggregate and repeated across each year. This view collapses to one row per profession and ranks the highest compound growth rates.

In [3]:
top_cagr = (
    growth_df
    .filter(pl.col('sector') == 'All')
    .group_by('profession')
    .agg(pl.col('cagr').drop_nulls().first().alias('cagr'))
    .with_columns((pl.col('cagr') * 100).alias('cagr_pct'))
    .sort('cagr', descending=True)
)
top_cagr.head(5)

profession,cagr,cagr_pct
cat,f64,f64
"""physiotherapists""",0.077005,7.7005
"""pharmacists""",0.069605,6.960519
"""doctors""",0.057174,5.717358
"""nurses""",0.056537,5.653739


## Nurse YoY Growth Table

PS-001 identified nurses as the top-growth profession. This table shows profession-level year-on-year growth for nurses using the aggregate 'All' sector.

In [4]:
nurse_yoy = (
    growth_df
    .filter((pl.col('profession') == 'nurses') & (pl.col('sector') == 'All'))
    .select(['profession', 'sector', 'year', 'yoy_pct', 'cagr'])
    .sort('year')
)
nurse_yoy

profession,sector,year,yoy_pct,cagr
cat,cat,i32,f64,f64
"""nurses""","""All""",2006,null,0.056537
"""nurses""","""All""",2007,6.713815,0.056537
"""nurses""","""All""",2008,8.404979,0.056537
"""nurses""","""All""",2009,10.669586,0.056537
"""nurses""","""All""",2010,9.510302,0.056537
…,…,…,…,…
"""nurses""","""All""",2015,3.687065,0.056537
"""nurses""","""All""",2016,3.989232,0.056537
"""nurses""","""All""",2017,2.167106,0.056537


## Save Static Figures

Save a profession-level CAGR bar chart and a profession-level YoY growth line chart for downstream reporting and dashboard integration.

In [5]:
cagr_plot = top_cagr.sort('cagr')
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(cagr_plot['profession'].to_list(), cagr_plot['cagr_pct'].to_list(), color='#2A6F97')
ax.set_title('Compound Annual Growth Rate by Profession')
ax.set_xlabel('CAGR (%)')
ax.set_ylabel('Profession')
fig.tight_layout()
fig.savefig(cagr_figure_path, bbox_inches='tight')
plt.close(fig)

yoy_plot = (
    growth_df
    .filter(pl.col('sector') == 'All')
    .select(['profession', 'year', 'yoy_pct'])
    .sort(['profession', 'year'])
)

fig, ax = plt.subplots(figsize=(9, 5))
for profession in yoy_plot['profession'].unique().sort().to_list():
    profession_slice = yoy_plot.filter(pl.col('profession') == profession)
    ax.plot(
        profession_slice['year'].to_list(),
        profession_slice['yoy_pct'].to_list(),
        marker='o',
        linewidth=2,
        label=profession,
    )

ax.set_title('Year-on-Year Growth by Profession')
ax.set_xlabel('Year')
ax.set_ylabel('YoY Growth (%)')
ax.legend(title='Profession')
fig.tight_layout()
fig.savefig(yoy_figure_path, bbox_inches='tight')
plt.close(fig)

print('Saved:', cagr_figure_path)
print('Saved:', yoy_figure_path)

Saved: /Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/artifacts/ps-003-workforce-growth-analysis/reports/figures/cagr_by_profession.png
Saved: /Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/artifacts/ps-003-workforce-growth-analysis/reports/figures/yoy_growth_by_profession.png
